# Clean & Build Dataset

**Goal**: Gather all features from notebooks, impute missing values, and build
a clean analytical dataset ready for modeling.

**Target**: `calculated_overdue`

In [5]:
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
from datetime import datetime
from sqlalchemy import create_engine, text

load_dotenv()
DB_URL = os.getenv('DB_URL')
engine = create_engine(DB_URL)

def q(sql):
    
    return pd.read_sql(sql, engine)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

## 1. Base Tasks with Target & Derived Features

In [6]:
base = q("""
    SELECT
        id,
        status,
        approval_status,
        lead_approval_status,
        weight_level,
        is_planned::int AS is_planned,
        risk_mapping,
        start_date,
        end_date,
        actual_end_date,
        created_date,
        updated_date,
        major_activity_id,
        created_by_id,
        position_id,
        department_id,
        CASE WHEN derived_from_cross_department_assignment_id IS NOT NULL THEN 1 ELSE 0 END AS is_cross_dept
    FROM tasks_task
""")
print(f'Base tasks: {len(base)}')

Base tasks: 13895


In [7]:
# Convert all date columns to tz-naive datetime64
# PostgreSQL timestamptz arrives as object dtype (Python datetime), not datetimetz
for col in ['start_date', 'end_date', 'actual_end_date', 'created_date', 'updated_date']:
    ser = pd.to_datetime(base[col], errors='coerce')
    if hasattr(ser.dt, 'tz') and ser.dt.tz is not None:
        base[col] = ser.dt.tz_localize(None)
    else:
        base[col] = ser
print('All date columns converted to tz-naive datetime64')


All date columns converted to tz-naive datetime64


### Handling Missing `actual_end_date` — Three-Tier Approach

**The problem:** 5,658 completed tasks (43% of all completed) have a NULL `actual_end_date`.
The old v1 logic silently treated these as “not overdue” (0), which undercounts the true overdue rate.

**The fix:** We use a three-tier approach to infer the actual completion date:

---
#### Tier 1: `actual_end_date` exists (ground truth)
If the task has an `actual_end_date`, compare it directly to `end_date`.
```
actual_end_date > end_date  ->  OVERDUE
actual_end_date <= end_date ->  on time
```

#### Tier 2: No `actual_end_date`, but history has a completion record
The `tasks_task_history` table stores a full snapshot every time a task is changed.
We find the **first** time the task’s `status` was set to `‘completed’` and use
that `history_date` as the inferred completion date.
```
first_completed_at = MIN(history_date) WHERE status = 'completed'
first_completed_at > end_date  ->  OVERDUE
```
This recovers **231 tasks** with high confidence.

#### Tier 3: No `actual_end_date`, no history at all
5,427 tasks have **zero rows in the history table** (bulk-imported directly into
the database, bypassing the app’s audit trail). For these, we use `updated_date`
as the best available proxy.
```
updated_date > end_date  ->  OVERDUE
```
**Caveat:** `updated_date` may reflect any edit (comment, description change),
not just completion. This is a low-confidence approximation.

---
#### Tracking confidence with `target_source`
Each row gets a `target_source` column so you know how its label was determined:
- `actual_end_date` — high confidence (Tier 1)
- `history_completion` — high confidence (Tier 2)
- `updated_date` — low confidence (Tier 3)
- `open_task` — non-completed task past deadline
- `status_based` — archived, terminated, or within deadline

All date comparisons use a **fixed cutoff** (`2026-07-14`) instead of `CURRENT_DATE`
to ensure labels are reproducible across runs.

In [8]:
# Step 1: Load first completion timestamps from history table
# For Tier 2: find the first time each task was marked 'completed' in the audit log.
first_completed = q("""
    SELECT
        history_relation_id AS task_id,
        MIN(history_date) AS first_completed_at
    FROM tasks_task_history
    WHERE status = 'completed'
    GROUP BY history_relation_id
""")
first_completed['first_completed_at'] = first_completed['first_completed_at'].dt.tz_localize(None)
base = base.merge(first_completed, left_on='id', right_on='task_id', how='left')
print(f'Tasks with a completion record in history: {first_completed["task_id"].nunique()}')

# Step 2: Build the three-tier completion date
# Tier 1 -> Tier 2 -> Tier 3: each fills in gaps left by the previous tier.
FIXED_CUTOFF = pd.Timestamp('2026-07-14').normalize()

# All datetime columns are already tz-naive (cleaned in cell above)
base['completion_date'] = base['actual_end_date'].copy()                    # Tier 1
base['completion_date'] = base['completion_date'].fillna(                   # Tier 2
    base['first_completed_at'].dt.normalize()
)
base['completion_date'] = base['completion_date'].fillna(                   # Tier 3
    base['updated_date'].dt.normalize()
)

# Step 3: Determine which tier/source was used for each row
conditions = [
    (base['status'] == 'completed') & (base['actual_end_date'].notna()),
    (base['status'] == 'completed') & (base['actual_end_date'].isna()) & (base['first_completed_at'].notna()),
    (base['status'] == 'completed') & (base['actual_end_date'].isna()) & (base['first_completed_at'].isna()),
    ~base['status'].isin(['completed', 'terminated', 'archived']) & (base['end_date'] < FIXED_CUTOFF),
]
source_labels = ['actual_end_date', 'history_completion', 'updated_date', 'open_task']
base['target_source'] = np.select(conditions, source_labels, default='status_based')
# Step 4: Compute the overdue flag
overdue_conditions = [
    (base['status'] == 'completed') & (base['completion_date'] > base['end_date']),
    ~base['status'].isin(['completed', 'terminated', 'archived']) & (base['end_date'] < FIXED_CUTOFF),
]
base['calculated_overdue'] = np.select(overdue_conditions, [1, 1], default=0)

print(f'Overdue rate: {base["calculated_overdue"].mean():.2%}')
print('Target source distribution:')
print(base['target_source'].value_counts())


Tasks with a completion record in history: 5327
Overdue rate: 47.81%
Target source distribution:
target_source
actual_end_date       7444
updated_date          5427
status_based           722
history_completion     231
open_task               71
Name: count, dtype: int64


## 2. Revisions (tasks_task_history)

Identical to notebook 02: revision_frequency uses **task age from created_date** (same denominator).

In [9]:
revisions = q("""
    SELECT
        history_relation_id AS task_id,
        COUNT(*) AS num_revisions,
        MAX(history_date) AS last_revision
    FROM tasks_task_history
    WHERE history_relation_id IS NOT NULL
    GROUP BY history_relation_id
""")

revisions['last_revision'] = revisions['last_revision'].dt.tz_localize(None)
# revision_frequency uses task age from created_date (same as notebook 02)
rev = revisions.copy()
task_age_map = (pd.Timestamp('2026-07-14').normalize() - base.set_index('id')['created_date']).dt.days
rev['task_age_days'] = rev['task_id'].map(task_age_map).fillna(0).clip(lower=1)
rev['revision_frequency'] = rev['num_revisions'] / rev['task_age_days']
rev['revision_recency'] = (pd.Timestamp('2026-07-14').normalize() - rev['last_revision']).dt.days

base = base.merge(rev[['task_id', 'num_revisions', 'revision_frequency', 'revision_recency']],
                  left_on='id', right_on='task_id', how='left')
base['num_revisions'] = base['num_revisions'].fillna(0).astype(int)
base['revision_frequency'] = base['revision_frequency'].fillna(0)
base['revision_recency'] = base['revision_recency'].fillna(0)
# base.drop(columns=['task_id'], inplace=True)

## 3. Subtasks (tasks_sub_task)

In [10]:
subtasks = q("""
    SELECT
        task_id,
        COUNT(*) AS num_subtasks,
        SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS num_completed_subtasks,
        SUM(CASE WHEN is_overdue = TRUE THEN 1 ELSE 0 END) AS num_overdue_subtasks
    FROM tasks_sub_task
    WHERE task_id IS NOT NULL
    GROUP BY task_id
""")
subtasks['subtask_completion_pct'] = (subtasks['num_completed_subtasks'] / subtasks['num_subtasks']).fillna(0)
subtasks['subtask_overdue_rate'] = (subtasks['num_overdue_subtasks'] / subtasks['num_subtasks']).fillna(0)

base = base.merge(subtasks[['task_id', 'num_subtasks', 'subtask_completion_pct', 'subtask_overdue_rate']],
                  left_on='id', right_on='task_id', how='left')
base['has_subtasks'] = (base['num_subtasks'] > 0).astype(int)
base['num_subtasks'] = base['num_subtasks'].fillna(0).astype(int)
base['subtask_completion_pct'] = base['subtask_completion_pct'].fillna(0)
base['subtask_overdue_rate'] = base['subtask_overdue_rate'].fillna(0)
base.drop(columns=['task_id'], inplace=True)

## 3b. Subtask Completion % at Halfway (tasks_sub_task_history)

Uses the history table to determine each subtask's status at the **halfway point**
(`start_date + planned_duration/2`). Falls back to current status if no history entry
exists before halfway.

In [11]:
halfway_sub = q("""
WITH task_halfway AS (
    SELECT id AS task_id,
           start_date + (end_date - start_date) / 2 AS halfway_date
    FROM tasks_task
    WHERE start_date IS NOT NULL AND end_date IS NOT NULL AND start_date <= end_date
),
latest_history AS (
    SELECT DISTINCT ON (sth.id)
        sth.id AS sub_task_id,
        sth.status AS status_at_halfway
    FROM tasks_sub_task_history sth
    JOIN tasks_sub_task st ON st.id = sth.id
    JOIN task_halfway th ON th.task_id = st.task_id
    WHERE sth.history_date <= th.halfway_date
    ORDER BY sth.id, sth.history_date DESC
)
SELECT
    st.task_id,
    COUNT(*)::int AS num_subtasks_at_halfway,
    COUNT(*) FILTER (WHERE COALESCE(lh.status_at_halfway, st.status) = 'completed')::int
        AS num_completed_at_halfway
FROM tasks_sub_task st
JOIN task_halfway th ON th.task_id = st.task_id
LEFT JOIN latest_history lh ON lh.sub_task_id = st.id
WHERE st.created_date <= th.halfway_date
GROUP BY st.task_id
""")
halfway_sub['subtask_completion_pct_at_halfway'] = (
    halfway_sub['num_completed_at_halfway'] / halfway_sub['num_subtasks_at_halfway']
).fillna(0)

base = base.merge(halfway_sub[['task_id', 'subtask_completion_pct_at_halfway']],
                  left_on='id', right_on='task_id', how='left')
base['subtask_completion_pct_at_halfway'] = base['subtask_completion_pct_at_halfway'].fillna(0)
base.drop(columns=['task_id'], inplace=True)

## 4. Challenges (tasks_task_challenge_groups)

In [12]:
challenges = q("""
    SELECT
        tcg.task_id,
        COUNT(*) AS num_challenges
    FROM tasks_task_challenge_groups tcg
    WHERE tcg.task_id IS NOT NULL
    GROUP BY tcg.task_id
""")
challenges['has_challenges'] = 1

base = base.merge(challenges[['task_id', 'num_challenges', 'has_challenges']],
                  left_on='id', right_on='task_id', how='left')
base['num_challenges'] = base['num_challenges'].fillna(0).astype(int)
base['has_challenges'] = base['has_challenges'].fillna(0).astype(int)
base.drop(columns=['task_id'], inplace=True)

## 5. Department Resolution & Past Overdue Rate

Resolution cascade matches notebook 02: `position.department_id → task.department_id` (2-level).
Aggregation uses the same 2-level key — no mismatch.

In [13]:
# Department-level aggregates — uses same 2-level cascade as resolution
dept_agg = q("""
    SELECT
        COALESCE(p.department_id, t.department_id) AS dept_id,
        COUNT(*) AS dept_task_count,
        AVG(CASE
            WHEN t.status = 'completed' AND t.actual_end_date IS NOT NULL AND t.actual_end_date > t.end_date THEN 1
            WHEN t.status NOT IN ('completed', 'terminated', 'archived') AND t.end_date < '2026-07-14'::date THEN 1
            ELSE 0
        END) AS dept_past_overdue_rate,
        AVG(COALESCE(rev_cnt.num_revisions, 0)) AS dept_avg_revisions
    FROM tasks_task t
    LEFT JOIN basedata_position p ON p.id = t.position_id
    LEFT JOIN (
        SELECT history_relation_id AS task_id, COUNT(*) AS num_revisions
        FROM tasks_task_history WHERE history_relation_id IS NOT NULL GROUP BY history_relation_id
    ) rev_cnt ON rev_cnt.task_id = t.id
    GROUP BY COALESCE(p.department_id, t.department_id)
""")

# Resolve department — 2-level cascade (same key as agg)
base['resolved_dept_id'] = base['position_id'].map(
    q("SELECT id AS position_id, department_id FROM basedata_position").set_index('position_id')['department_id']
).fillna(base['department_id'])

base = base.merge(dept_agg, left_on='resolved_dept_id', right_on='dept_id', how='left')
base.drop(columns=['dept_id'], inplace=True)

## 6. Employee Past Overdue Rate

Matches notebook 02: computed per **assignee** (via position.user_id), merged on assignee_id.

In [14]:
emp_agg = q("""
    SELECT
        p.user_id AS assignee_id,
        AVG(CASE
            WHEN t.status = 'completed' AND t.actual_end_date IS NOT NULL AND t.actual_end_date > t.end_date THEN 1
            WHEN t.status NOT IN ('completed', 'terminated', 'archived') AND t.end_date < '2026-07-14'::date THEN 1
            ELSE 0
        END) AS emp_past_overdue_rate
    FROM tasks_task t
    LEFT JOIN basedata_position p ON p.id = t.position_id
    WHERE p.user_id IS NOT NULL
    GROUP BY p.user_id
""")

# Map task -> position -> user (assignee), matching notebook 02
task_assignee = q("""
    SELECT t.id AS task_id, p.user_id AS assignee_id
    FROM tasks_task t
    LEFT JOIN basedata_position p ON p.id = t.position_id
""")
assignee_map = task_assignee.set_index('task_id')['assignee_id']
base['assignee_id'] = base['id'].map(assignee_map)

base = base.merge(emp_agg, on='assignee_id', how='left')
base.drop(columns=['assignee_id'], inplace=True)

## 7. Position Past Overdue Rate

In [15]:
pos_agg = q("""
    SELECT
        t.position_id,
        AVG(CASE
            WHEN t.status = 'completed' AND t.actual_end_date IS NOT NULL AND t.actual_end_date > t.end_date THEN 1
            WHEN t.status NOT IN ('completed', 'terminated', 'archived') AND t.end_date < '2026-07-14'::date THEN 1
            ELSE 0
        END) AS pos_past_overdue_rate
    FROM tasks_task t
    WHERE t.position_id IS NOT NULL
    GROUP BY t.position_id
""")

base = base.merge(pos_agg, on='position_id', how='left')

## 8. Cross-Department Assignment Flag

In [16]:
cross_dept = q("""
    SELECT t.id AS task_id, 1 AS cross_dept_pair_exists
    FROM tasks_task t
    JOIN tasks_cross_department_assignments cda ON cda.id = t.derived_from_cross_department_assignment_id
""")

base = base.merge(cross_dept, left_on='id', right_on='task_id', how='left')
base['cross_dept_pair_exists'] = base['cross_dept_pair_exists'].fillna(0).astype(int)
base.drop(columns=['task_id'], inplace=True)

## 9. Major Activity (status + approval_status)

In [17]:
ma_info = q("""
    SELECT
        ma.id AS major_activity_id,
        ma.status AS ma_status,
        ma.approval_status AS ma_approval_status,
        ma.kpi_id
    FROM tasks_major_activity ma
""")

base = base.merge(ma_info, on='major_activity_id', how='left')

## 10. Count of MA Revisions (tasks_major_activity_history)

In [18]:
ma_revisions = q("""
    SELECT id AS major_activity_id, COUNT(*) AS num_ma_revisions
    FROM tasks_major_activity_history
    WHERE id IS NOT NULL
    GROUP BY id
""")

base = base.merge(ma_revisions, on='major_activity_id', how='left')
base['num_ma_revisions'] = base['num_ma_revisions'].fillna(0).astype(int)

## 11. KPI Features (via MA -> KPI)

In [19]:
kpi_features = q("""
    SELECT
        kpi.id AS kpi_id,
        kpi.is_overdue AS kpi_is_overdue,
        kpi.status AS kpi_status
    FROM tasks_kpi kpi
""")
kpi_features['kpi_is_overdue_flag'] = kpi_features['kpi_is_overdue'].astype(int)
status_order = {'not_started': 0, 'ongoing': 1, 'completed': 2, 'terminated': 3, 'archived': 4}
kpi_features['kpi_status_ordinal'] = kpi_features['kpi_status'].map(status_order).fillna(1)

base = base.merge(kpi_features[['kpi_id', 'kpi_is_overdue_flag', 'kpi_status_ordinal']],
                  on='kpi_id', how='left')
base['kpi_is_overdue_flag'] = base['kpi_is_overdue_flag'].fillna(0).astype(int)
base['kpi_status_ordinal'] = base['kpi_status_ordinal'].fillna(1).astype(int)

## 12. KPI History (num_kpi_revisions)

In [20]:
kpi_hist = q("""
    SELECT id AS kpi_id, COUNT(*) AS num_kpi_revisions
    FROM tasks_kpi_history
    WHERE id IS NOT NULL
    GROUP BY id
""")

base = base.merge(kpi_hist, on='kpi_id', how='left')
base['num_kpi_revisions'] = base['num_kpi_revisions'].fillna(0).astype(int)

## 13. Sub-Task Challenge Groups

In [21]:
sub_chal = q("""
    SELECT stcg.subtask_id AS sub_task_id, st.task_id
    FROM tasks_sub_task_challenge_groups stcg
    JOIN tasks_sub_task st ON st.id = stcg.subtask_id
""")

task_sub_chal = sub_chal.groupby('task_id').agg(
    has_subtask_challenge=('sub_task_id', lambda x: 1),
    num_subtask_challenges=('sub_task_id', 'count')
).reset_index()

base = base.merge(task_sub_chal, left_on='id', right_on='task_id', how='left')
base['has_subtask_challenge'] = base['has_subtask_challenge'].fillna(0).astype(int)
base['num_subtask_challenges'] = base['num_subtask_challenges'].fillna(0).astype(int)
base.drop(columns=['task_id'], inplace=True)

## 14. KPI Challenge Groups (actual)

In [22]:
kpi_chal = q("""
    SELECT kpi_id, COUNT(*) AS num_kpi_challenges
    FROM tasks_kpis_challegne_groups GROUP BY kpi_id
""")
kpi_chal['has_kpi_challenge'] = 1

base = base.merge(kpi_chal, on='kpi_id', how='left')
base['has_kpi_challenge'] = base['has_kpi_challenge'].fillna(0).astype(int)
base['num_kpi_challenges'] = base['num_kpi_challenges'].fillna(0).astype(int)

## 15. KPI Potential Challenge Groups

In [23]:
kpi_pot = q("""
    SELECT kpi_id, COUNT(*) AS num_kpi_potential_challenges
    FROM tasks_kpis_potential_challenge_groups GROUP BY kpi_id
""")
kpi_pot['has_kpi_potential_challenge'] = 1

base = base.merge(kpi_pot, on='kpi_id', how='left')
base['has_kpi_potential_challenge'] = base['has_kpi_potential_challenge'].fillna(0).astype(int)
base['num_kpi_potential_challenges'] = base['num_kpi_potential_challenges'].fillna(0).astype(int)

## 16. Comments (KPI, MA, Task)

In [24]:
comments = q("""
    SELECT object_id, content_type_id
    FROM comments_comment
""")

# KPI comments (content_type_id=22)
kpi_cmt = comments[comments['content_type_id'] == 22].groupby('object_id').size().reset_index(name='kpi_comment_count')
kpi_cmt.rename(columns={'object_id': 'kpi_id'}, inplace=True)
base = base.merge(kpi_cmt, on='kpi_id', how='left')
base['kpi_comment_count'] = base['kpi_comment_count'].fillna(0).astype(int)

# MA comments (content_type_id=23)
ma_cmt = comments[comments['content_type_id'] == 23].groupby('object_id').size().reset_index(name='ma_comment_count')
ma_cmt.rename(columns={'object_id': 'major_activity_id'}, inplace=True)
base = base.merge(ma_cmt, on='major_activity_id', how='left')
base['ma_comment_count'] = base['ma_comment_count'].fillna(0).astype(int)

# Task comments (content_type_id=24)
task_cmt = comments[comments['content_type_id'] == 24].groupby('object_id').size().reset_index(name='task_comment_count')
base = base.merge(task_cmt, left_on='id', right_on='object_id', how='left')
base['task_comment_count'] = base['task_comment_count'].fillna(0).astype(int)
base.drop(columns=['object_id'], inplace=True)

## 17. Sub-Task History (avg status changes)

In [25]:
sub_hist = q("""
    SELECT sth.id AS sub_task_id, COUNT(DISTINCT sth.status) AS num_status_changes
    FROM tasks_sub_task_history sth
    WHERE sth.id IS NOT NULL
    GROUP BY sth.id
""")

st_map = q("""SELECT id AS sub_task_id, task_id FROM tasks_sub_task WHERE task_id IS NOT NULL""")
task_sub_churn = sub_hist.merge(st_map, on='sub_task_id', how='left')
task_sub_churn_agg = task_sub_churn.groupby('task_id')['num_status_changes'].mean().reset_index(name='avg_sub_status_changes')

base = base.merge(task_sub_churn_agg, left_on='id', right_on='task_id', how='left')
base['avg_sub_status_changes'] = base['avg_sub_status_changes'].fillna(0)
base.drop(columns=['task_id'], inplace=True)

## 18. Imputation

In [26]:
null_cols = [
    'is_cross_dept', 'revision_frequency', 'revision_recency',
    'subtask_completion_pct', 'subtask_overdue_rate',
    'subtask_completion_pct_at_halfway',
    'num_challenges', 'has_challenges',
    'dept_past_overdue_rate', 'dept_avg_revisions',
    'emp_past_overdue_rate', 'pos_past_overdue_rate',
    'cross_dept_pair_exists',
    'kpi_is_overdue_flag', 'kpi_status_ordinal',
    'has_subtask_challenge', 'num_subtask_challenges',
    'has_kpi_challenge', 'num_kpi_challenges',
    'has_kpi_potential_challenge', 'num_kpi_potential_challenges',
    'kpi_comment_count', 'ma_comment_count', 'task_comment_count',
    'avg_sub_status_changes',
]

null_summary = []
for col in null_cols:
    if col in base.columns:
        null_pct = base[col].isna().mean()
        null_summary.append({'Feature': col, 'Null %': f'{null_pct:.1%}'})
        
null_summary = pd.DataFrame(null_summary)
null_summary

,Feature,Null %
0,is_cross_dept,0.0%
1,revision_frequency,0.0%
2,revision_recency,0.0%
3,subtask_completion_pct,0.0%
4,subtask_overdue_rate,0.0%
5,subtask_completion_pct_at_halfway,0.0%
6,num_challenges,0.0%
7,has_challenges,0.0%
8,dept_past_overdue_rate,0.0%
9,dept_avg_revisions,0.0%


In [27]:
# Fill 0 — absence means no signal
fill_zero_cols = [
    'is_cross_dept', 'revision_frequency', 'revision_recency',
    'subtask_completion_pct', 'subtask_overdue_rate',
    'subtask_completion_pct_at_halfway',
    'num_challenges', 'has_challenges',
    'cross_dept_pair_exists',
    'kpi_is_overdue_flag',
    'has_subtask_challenge', 'num_subtask_challenges',
    'has_kpi_challenge', 'num_kpi_challenges',
    'has_kpi_potential_challenge', 'num_kpi_potential_challenges',
    'kpi_comment_count', 'ma_comment_count', 'task_comment_count',
    'avg_sub_status_changes',
]
for col in fill_zero_cols:
    if col in base.columns:
        base[col] = base[col].fillna(0)

# Fill 1 for kpi_status_ordinal (ongoing is the middle/neutral state)
if 'kpi_status_ordinal' in base.columns:
    base['kpi_status_ordinal'] = base['kpi_status_ordinal'].fillna(1)

# Fill global mean for unobserved groups
for col in ['dept_past_overdue_rate', 'dept_avg_revisions', 'emp_past_overdue_rate', 'pos_past_overdue_rate']:
    if col in base.columns:
        base[col] = base[col].fillna(base[col].mean())

# position_id add Unknown
base['position_id'] = base['position_id'].fillna('UNKNOWN')

print('Imputation complete.')
remaining_nulls = base[[c for c in null_cols if c in base.columns]].isna().sum().sum()
print(f'Remaining nulls in feature columns: {remaining_nulls}')

Imputation complete.
Remaining nulls in feature columns: 0


## 19. Encoding

In [28]:
# Ordinal encoding
status_order = {'not_started': 0, 'ongoing': 1, 'in_progress': 1, 'completed': 2,
               'terminated': 3, 'archived': 4}
base['status_encoded'] = base['status'].map(status_order).fillna(1)

apr_order = {'pending': 0, 'in_review': 1, 'approved': 2, 'rejected': 3}
base['approval_status_encoded'] = base['approval_status'].map(apr_order).fillna(1)
base['lead_approval_status_encoded'] = base['lead_approval_status'].map(apr_order).fillna(1)

ma_status_order = {'not_started': 0, 'ongoing': 1, 'completed': 2, 'terminated': 3}
base['ma_status_encoded'] = base['ma_status'].map(ma_status_order).fillna(1)

ma_apr_order = {'pending': 0, 'in_review': 1, 'approved': 2, 'rejected': 3}
base['ma_approval_status_encoded'] = base['ma_approval_status'].map(ma_apr_order).fillna(1)

# One-hot for weight_level
wl_dummies = pd.get_dummies(base['weight_level'], prefix='wl')
base = pd.concat([base, wl_dummies], axis=1)

# Target encoding for position_id
pos_rate = base.groupby('position_id')['calculated_overdue'].mean()
base['position_id_encoded'] = base['position_id'].map(pos_rate)

print(f'After encoding: {base.shape[1]} cols')

After encoding: 66 cols


## 20. Final Dataset

In [29]:
final_features = [
    'status_encoded', 'approval_status_encoded', 'lead_approval_status_encoded',
    'ma_status_encoded', 'ma_approval_status_encoded',
    'planned_duration', 'creation_to_planned_start',
    'created_dow', 'created_is_weekend', 'created_is_friday',
    'created_month', 'created_quarter',
    'days_since_update',
    'is_planned', 'risk_mapping',
    'num_revisions', 'revision_frequency', 'revision_recency',
    'num_subtasks', 'has_subtasks',
    'subtask_completion_pct', 'subtask_overdue_rate',
    'subtask_completion_pct_at_halfway',
    'num_challenges', 'has_challenges',
    'num_ma_revisions',
    *[c for c in base.columns if c.startswith('wl_')],
    'is_cross_dept',
    'cross_dept_pair_exists',
    'kpi_is_overdue_flag', 'kpi_status_ordinal',
    'num_kpi_revisions',
    'has_subtask_challenge', 'num_subtask_challenges',
    'has_kpi_challenge', 'num_kpi_challenges',
    'has_kpi_potential_challenge', 'num_kpi_potential_challenges',
    'kpi_comment_count', 'ma_comment_count', 'task_comment_count',
    'avg_sub_status_changes',
    'dept_past_overdue_rate', 'dept_avg_revisions',
    'emp_past_overdue_rate', 'pos_past_overdue_rate',
    'position_id_encoded',
]

final_features = [c for c in final_features if c in base.columns]
dataset = base[['id', 'calculated_overdue'] + final_features].copy()

print(f'Final dataset shape: {dataset.shape}')
print(f'Features: {len(final_features)}')
print(f'Target distribution:\n{dataset["calculated_overdue"].value_counts(normalize=True).to_string()}')

Final dataset shape: (13895, 43)
Features: 41
Target distribution:
calculated_overdue
0    0.521914
1    0.478086


In [30]:
nulls = dataset.isna().sum()
nulls = nulls[nulls > 0]
if len(nulls) > 0:
    print('WARNING: Remaining nulls:')
    print(nulls)
else:
    print('Dataset is clean — 0 nulls remaining.')

Dataset is clean — 0 nulls remaining.


In [31]:
dataset.head()

,id,calculated_overdue,status_encoded,approval_status_encoded,lead_approval_status_encoded,ma_status_encoded,ma_approval_status_encoded,is_planned,risk_mapping,num_revisions,revision_frequency,revision_recency,num_subtasks,has_subtasks,subtask_completion_pct,subtask_overdue_rate,subtask_completion_pct_at_halfway,num_challenges,has_challenges,num_ma_revisions,wl_high,wl_low,wl_mid,is_cross_dept,cross_dept_pair_exists,kpi_is_overdue_flag,kpi_status_ordinal,num_kpi_revisions,has_subtask_challenge,num_subtask_challenges,has_kpi_challenge,num_kpi_challenges,has_kpi_potential_challenge,num_kpi_potential_challenges,kpi_comment_count,ma_comment_count,task_comment_count,avg_sub_status_changes,dept_past_overdue_rate,dept_avg_revisions,emp_past_overdue_rate,pos_past_overdue_rate,position_id_encoded
0,dcf0526e-9ebf-47e1-af13-fe2fad6fc713,0,3.0,0,2.0,0.0,0,1,0.0,0,0.000000,0.0,4,1,0.0,0.0,0.0,0,0,2,True,False,False,0,0,1,1,1,0,0,0,0,0,0,0,0,0,1.25,0.168529,1.567505,0.088962,0.088962,0.423394
1,913fb057-fecd-452c-837c-9cd1ec938aa2,1,2.0,0,2.0,2.0,2,1,6.0,0,0.000000,0.0,0,0,0.0,0.0,0.0,0,0,18,False,False,True,0,0,1,1,24,0,0,0,0,0,0,0,0,1,0.00,0.328366,0.609272,0.188235,0.188235,0.762745
2,549dfafd-35f9-42a3-b76d-3d63720c91e0,1,2.0,0,2.0,2.0,2,0,6.0,3,0.018405,163.0,0,0,0.0,0.0,0.0,0,0,67,False,False,True,0,0,1,1,18,0,0,0,0,0,0,0,0,0,0.00,0.328366,0.609272,0.188235,0.188235,0.762745
3,fdb4148a-9bb0-48c1-a8d0-8e4a5a13ca91,1,2.0,0,2.0,2.0,0,1,6.0,3,0.015873,178.0,0,0,0.0,0.0,0.0,0,0,60,False,False,True,1,1,0,2,32,0,0,0,0,1,3,1,0,0,0.00,0.209412,2.020588,0.250660,0.250660,0.493404
4,d0b7134a-f177-41ef-a05b-bda006450edc,0,2.0,0,2.0,2.0,0,1,0.0,4,0.024242,164.0,0,0,0.0,0.0,0.0,0,0,30,True,False,False,0,0,0,2,29,0,0,0,0,1,1,0,0,0,0.00,0.133913,5.620870,0.202884,0.199531,0.531627


## 21. Prediction-Time Feature Availability

Define which features are available at each prediction point:
- **At assignment (creation)**: features known before the task starts
- **At halfway** (start + planned_duration/2): adds features that accumulate during execution

`subtask_completion_pct_at_halfway` is computed from `tasks_sub_task_history` with a
cutoff at the halfway date — it reflects the true completion rate at the prediction point.

In [32]:
# Features available at task creation / assignment time
features_at_assignment = [
    'status_encoded',
    'approval_status_encoded', 'lead_approval_status_encoded',
    'ma_status_encoded', 'ma_approval_status_encoded',
    'planned_duration', 'creation_to_planned_start',
    'created_dow', 'created_is_weekend', 'created_is_friday',
    'created_month', 'created_quarter',
    'is_planned', 'risk_mapping',
    *[c for c in dataset.columns if c.startswith('wl_')],
    'is_cross_dept', 'cross_dept_pair_exists',
    'num_ma_revisions',
    'kpi_is_overdue_flag', 'kpi_status_ordinal',
    'num_kpi_revisions',
    'has_kpi_challenge', 'num_kpi_challenges',
    'has_kpi_potential_challenge', 'num_kpi_potential_challenges',
    'kpi_comment_count', 'ma_comment_count',
    'dept_past_overdue_rate', 'dept_avg_revisions',
    'emp_past_overdue_rate', 'pos_past_overdue_rate',
    'position_id_encoded',
]

# Additional features available at the halfway point
features_added_at_halfway = [
    'days_since_update',
    'num_revisions', 'revision_frequency', 'revision_recency',
    'num_subtasks', 'has_subtasks',
    'subtask_completion_pct', 'subtask_overdue_rate',
    'subtask_completion_pct_at_halfway',
    'num_challenges', 'has_challenges',
    'has_subtask_challenge', 'num_subtask_challenges',
    'avg_sub_status_changes',
    'task_comment_count',
]

features_at_halfway = features_at_assignment + features_added_at_halfway

# Filter to only existing columns
features_at_assignment = [c for c in features_at_assignment if c in dataset.columns]
features_at_halfway = [c for c in features_at_halfway if c in dataset.columns]

print(f'Features at assignment:     {len(features_at_assignment)}')
print(f'Features at halfway:        {len(features_at_halfway)}')

Features at assignment:     27
Features at halfway:        41


In [33]:
dataset_at_assignment = dataset[['id', 'calculated_overdue'] + features_at_assignment].copy()
dataset_at_halfway = dataset[['id', 'calculated_overdue'] + features_at_halfway].copy()

print(f'Assignment dataset shape: {dataset_at_assignment.shape}')
print(f'Halfway dataset shape:    {dataset_at_halfway.shape}')
print(f'Overlap: {len(set(features_at_assignment) & set(features_at_halfway))} shared features')
print(f'Halfway-only: {len(features_at_halfway) - len(features_at_assignment)} additional features')

Assignment dataset shape: (13895, 29)
Halfway dataset shape:    (13895, 43)
Overlap: 27 shared features
Halfway-only: 14 additional features


In [34]:
dataset_at_assignment.head()

,id,calculated_overdue,status_encoded,approval_status_encoded,lead_approval_status_encoded,ma_status_encoded,ma_approval_status_encoded,is_planned,risk_mapping,wl_high,wl_low,wl_mid,is_cross_dept,cross_dept_pair_exists,num_ma_revisions,kpi_is_overdue_flag,kpi_status_ordinal,num_kpi_revisions,has_kpi_challenge,num_kpi_challenges,has_kpi_potential_challenge,num_kpi_potential_challenges,kpi_comment_count,ma_comment_count,dept_past_overdue_rate,dept_avg_revisions,emp_past_overdue_rate,pos_past_overdue_rate,position_id_encoded
0,dcf0526e-9ebf-47e1-af13-fe2fad6fc713,0,3.0,0,2.0,0.0,0,1,0.0,True,False,False,0,0,2,1,1,1,0,0,0,0,0,0,0.168529,1.567505,0.088962,0.088962,0.423394
1,913fb057-fecd-452c-837c-9cd1ec938aa2,1,2.0,0,2.0,2.0,2,1,6.0,False,False,True,0,0,18,1,1,24,0,0,0,0,0,0,0.328366,0.609272,0.188235,0.188235,0.762745
2,549dfafd-35f9-42a3-b76d-3d63720c91e0,1,2.0,0,2.0,2.0,2,0,6.0,False,False,True,0,0,67,1,1,18,0,0,0,0,0,0,0.328366,0.609272,0.188235,0.188235,0.762745
3,fdb4148a-9bb0-48c1-a8d0-8e4a5a13ca91,1,2.0,0,2.0,2.0,0,1,6.0,False,False,True,1,1,60,0,2,32,0,0,1,3,1,0,0.209412,2.020588,0.250660,0.250660,0.493404
4,d0b7134a-f177-41ef-a05b-bda006450edc,0,2.0,0,2.0,2.0,0,1,0.0,True,False,False,0,0,30,0,2,29,0,0,1,1,0,0,0.133913,5.620870,0.202884,0.199531,0.531627


## 22. Export CSV Files

Save both prediction-time datasets at the project root.

In [35]:
import os
ROOT = os.path.dirname(os.path.abspath('.'))  # project root (parent of notebooks/)
dataset_at_assignment.to_csv(os.path.join(ROOT, 'dataset_at_creation.csv'), index=False)
dataset_at_halfway.to_csv(os.path.join(ROOT, 'dataset_at_halfway.csv'), index=False)
print(f'Saved dataset_at_creation.csv ({len(dataset_at_assignment)} rows × {dataset_at_assignment.shape[1]} cols)')
print(f'Saved dataset_at_halfway.csv  ({len(dataset_at_halfway)} rows × {dataset_at_halfway.shape[1]} cols)')

Saved dataset_at_creation.csv (13895 rows × 29 cols)
Saved dataset_at_halfway.csv  (13895 rows × 43 cols)
